[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/flame/flame.ipynb)

# Cellular Flame

This example computes a cellular flame: flames spreading from sparks through a lean mixture of fuel and air, merging into one front that breaks into cells as it grows. A premixed flame is a thin front where fuel burns, moving into the unburnt mixture as the heat it conducts ahead ignites the fuel there, and burning grows so steeply with temperature, as Arrhenius's law has it, that the front's speed hangs on how hot it burns. A part of the front bulging into the mixture sheds its heat into more of the mixture around it and draws its fuel from more of it. Where fuel spreads faster than heat, as hydrogen does through lean air, the fuel wins: the bulge burns hotter and bulges further, while the creases between bulges lose more heat than their fuel makes up, burn cooler and all but go out. The Lewis number, heat's diffusivity over fuel's, measures this, and below one a smooth front is unstable. The mixture's density is taken as constant, so that nothing flows: the gas's expansion as it burns, which unsettles a front in a way of its own, is left out, and what remains is the instability that diffusion alone makes.

The algebra is the plane joined with three degenerate directions, for the volume, the fuel and the heat a parcel of the mixture holds, `Algebra("x+y+") * Algebra("v0f0h0")`. A parcel is a point weighed by its volume and offset along the fuel's and the heat's directions by what it holds, which `mv.f &` and `mv.h &` read off it. Burning is then one fixed direction in that space of amounts, `reaction = heat - fuel`, and what burns in a step moves each parcel along it, `parcels + burnt * reaction`: the reaction's stoichiometry is an element of the algebra, not a rule kept between two separate fields. The plane `energy = mv.h + mv.f` reads the energy a parcel holds, its heat and the heat its fuel would release, and pairs with the reaction to nothing, `energy & reaction == 0`: burning moves energy from the fuel into the heat and makes none. Each amount spreads by its own drive, minus its gradient wedged with the amount, `-(gradients ^ mv.h)` for the heat, which a diffusivity maps to that amount's flux. Since burning leaves `energy & parcels` alone, that reading changes only where heat and fuel spread at different rates, and its picture shows the mechanism of the cells: energy gathering behind the bulges and draining from the creases.

In the notation of combustion theory, with $T$ the temperature's rise above the unburnt mixture's in units of the rise burning brings and $Y$ the fuel's share in units of the mean unburnt mixture's, the thermo-diffusive model reads $\partial_t T = \nabla^2 T + \omega$ and $\partial_t Y = \mathrm{Le}^{-1}\nabla^2 Y - \omega$, with the reaction rate $\omega = \frac{\beta^2}{2\,\mathrm{Le}}\,Y\exp\frac{\beta(T-1)}{1+\gamma(T-1)}$ for the Zeldovich number $\beta$ and the heat release $\gamma$. The enthalpy $H = T + Y$ obeys $\partial_t H = \nabla^2 H + (\mathrm{Le}^{-1} - 1)\nabla^2 Y$, and a flat front breaks into cells where $\beta(\mathrm{Le} - 1) < -2$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import jax
import numpy as np

from numga import Algebra, stack
from numga.backend.jax import JaxContext
from numga.sparse import spdiag
from examples.util.animation import show_animation
from examples.util.topology import Topology
from examples.thermodynamics.flame import render, scenarios

jax.config.update("jax_enable_x64", True)

# The plane, and the volume, the fuel and the heat a parcel holds: each a direction of its own, degenerate, since
# nothing measures a volume or an amount of fuel or of heat against a length.
ga = Algebra("x+y+") * Algebra("v0f0h0")
mv = JaxContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# The planes reading a place's run across the plane and up it.
Upright = ga.gatype.from_blades("x y")                                        # Vector+
Drive = ga.gatype.from_blades("xf yf xh yh")                                  # Bivector+
# The plane's two directions, across and up.
directions = stack((mv.x.dual(), mv.y.dual()))                                # [2] Point-
# A unit of volume, the weight of a place, a unit of fuel and a unit of heat.
volume = mv.v.dual()                                                          # [] Point-
fuel = mv.f.dual()                                                            # [] Point-
heat = mv.h.dual()                                                            # [] Point-
# The plane reading the energy a parcel holds: its heat, and the heat its fuel would release, a unit of heat for each
# unit of fuel.
energy = mv.h + mv.f                                                          # [] Plane+

## The disk and the sparks

The mixture fills a disk of triangles in staggered rings, nearly equilateral, so that no direction across the disk is favoured, and fine enough that the zone where the fuel burns spans more than a triangle. Lengths are in units of the flame's thickness, heat's diffusivity over the speed of a flat flame, and times in units of the time such a flame takes to cross its thickness. A corner's hat gradient, the plane through the facing side over twice the triangle's area, carries every spreading below.

Three sparks of different sizes, as if lit one after another, start three flames, their edges roughened by noise so that each front has something to break up from.

In [ ]:
def triangles(topology: Topology, vertices: Point) -> tuple[Scalar, Plane, Scalar]:
    """Each triangle's area `Scalar+[faces]`, each corner's hat gradient `[corners] Plane+[faces]`, which reads one
    from the facing side to the corner, and each vertex's share of the area around it `Scalar+[vertices]`."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)              # [corners] Point-[faces]
    # Twice a triangle's area: two of its sides joined, at unit volume, fuel and heat.
    double_areas = (((sides[0] & sides[1]) & volume) & fuel) & heat           # Scalar-[faces]
    areas = double_areas.norm() / 2                                           # Scalar+[faces]
    # A corner's hat gradient: the plane through the facing side, along volume, fuel and heat, over twice the area.
    gradients = (volume & fuel & heat & sides) / double_areas                 # [corners] Plane+[faces]
    return areas, gradients, (~topology.incidence[2, 0] * (areas / 3)).sum(axis=0)


# The disk, and its vertices, places of unit volume.
topology, coordinates = scenarios.disk()
vertices = (volume + mv(Upright, coordinates).dual()).field()                 # Point-[vertices]
areas, gradients, cells = triangles(topology, vertices)
# The share of the mixture burnt within the sparks, which holds its fuel's heat in place of its fuel, and the fuel the
# unburnt mixture holds around them.
burnt_share = mv.scalar(scenarios.hot_spot(coordinates)[:, None]).field()    # Scalar+[vertices]
parcels = volume + heat * burnt_share + fuel * (1 - burnt_share)              # Point-[vertices]
render.draw_start(topology, vertices, scenarios.radius, energy, parcels);

The three sparks, charred dark in the pale unburnt mixture, beside the energy each parcel holds, the same everywhere at the start: the sparks' parcels hold their fuel's heat in place of their fuel.

## Burning

Each unit of fuel burns at a rate that grows steeply with temperature, Arrhenius's law, the activation temperature over the temperature in an exponent. The Zeldovich number says how steeply near the burnt mixture's temperature, and the prefactor is the one that makes a flat flame move at unit speed where burning grows steeply. Over a step a parcel's fuel decays at that rate, and what burns moves the parcel along the reaction, a unit of heat released for each unit of fuel. The Lewis number is about lean hydrogen's in air, and the time step short enough that a step burns only a share of a hot parcel's fuel.

In [ ]:
# The mixture: its Lewis number, its Zeldovich number and the unburnt mixture's temperature, in units of the rise
# burning brings; the time step.
lewis, zeldovich, unburnt_temperature = scenarios.lewis, scenarios.zeldovich, scenarios.unburnt_temperature
dt = scenarios.dt
# Arrhenius's law in those units: the activation temperature, and the prefactor that makes a flat flame's speed one.
burnt_temperature = unburnt_temperature + 1
activation = zeldovich * burnt_temperature ** 2
prefactor = zeldovich ** 2 / (2 * lewis) * np.exp(activation / burnt_temperature)
# Burning a unit of fuel releases a unit of heat: the reaction's one direction in the space of amounts.
reaction = heat - fuel                                                        # [] Point-


def burnt(parcels: Point) -> Scalar:
    """The fuel each parcel burns over a step `Scalar+[vertices]`: its temperature, the unburnt mixture's raised by its
    heat, sets the rate each unit of its fuel burns at, and its fuel decays at that rate."""
    temperature = unburnt_temperature + (mv.h & parcels)                      # Scalar+[vertices]
    rates = prefactor * (-activation / temperature).exp()                     # Scalar+[vertices]
    return (mv.f & parcels) * (1 - (-dt * rates).exp())

## Spreading

Heat spreads down the temperature's gradient and fuel down its own amount's, the fuel faster by the inverse of the Lewis number. Each drive is minus a gradient wedged with the amount it pushes, and a diffusivity is a map from drives to fluxes, `Flux- <- Drive+`: each of the plane's two directions swept through the amount, read by the drive and driving along it. What a flux carries out of a vertex's cell, gathered over the triangles around it, makes for each amount a map from its field at the vertices to what leaves each cell. Both amounts spread explicitly: the time step burning asks for lies inside the limit that the fuel's faster spreading sets on triangles this small.

In [ ]:
corners = topology.incidence[2, 0]                                            # [corners] Scalar+[faces <- vertices]
# The temperature's drive at each corner's hat, minus its gradient wedged with the heat it pushes, and the fuel's,
# wedged with fuel.
heat_drives, fuel_drives = -(gradients ^ mv.h), -(gradients ^ mv.f)           # [corners] Drive+[faces], [corners] Drive+[faces]
# Heat's conductivity and fuel's diffusivity: each drive read across the plane's two directions, driving its amount
# along them.
heat_sweeps = directions & heat                                               # [2] Flux-
fuel_sweeps = directions & fuel                                               # [2] Flux-
conductivity = (heat_sweeps * (heat_sweeps & Drive)).sum(axis=0)              # [] Flux- <- Drive+
diffusivity = (fuel_sweeps * (fuel_sweeps & Drive)).sum(axis=0) / lewis       # [] Flux- <- Drive+
# What a flux carries out of each corner's cell: its hat gradient wedged with the flux, read for its heat or its fuel,
# gathered onto the vertices.
local = areas * (mv.h & (gradients[:, None] ^ conductivity(heat_drives[None, :])))   # [corners, corners] Scalar+[faces]
conduction = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]
local = areas * (mv.f & (gradients[:, None] ^ diffusivity(fuel_drives[None, :])))   # [corners, corners] Scalar+[faces]
diffusion = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]

## The flame

Each step every parcel burns along the reaction while its heat and its fuel spread, and JAX compiles the steps between two frames into one scan. The flame is drawn as a sheet of paper burns, beside the energy each parcel holds, `energy & parcels`. The frames start once the noise on the sparks' edges has smoothed out, and end as the front nears the disk's rim. This cell takes about a minute, most of it stepping the flame.

In [ ]:
def step(parcels: Point, _) -> tuple[Point, None]:
    """One step: what burns moves each parcel along the reaction, and what each amount's drive carries out of a cell
    leaves it."""
    conducted = -(conduction * (mv.h & parcels)) / cells                      # Scalar+[vertices]
    diffused = -(diffusion * (mv.f & parcels)) / cells                        # Scalar+[vertices]
    return parcels + burnt(parcels) * reaction + dt * (conducted * heat + diffused * fuel), None


@jax.jit
def strided(parcels: Point) -> Point:
    """The parcels a frame's steps later."""
    return jax.lax.scan(step, parcels, None, length=scenarios.n_stride)[0]


def flame(parcels: Point, n_frames: int) -> Iterator[tuple[Point, Scalar]]:
    """The parcels `Point-[vertices]`, and the rate their fuel burns at per volume `Scalar+[vertices]`, at each of
    `n_frames` frames from these."""
    for _ in range(n_frames):
        yield parcels, burnt(parcels) / dt
        parcels = strided(parcels)


frames = islice(flame(parcels, scenarios.n_frames), scenarios.n_unshown, None)
show_animation(render.animate(topology, vertices, scenarios.radius, energy, frames), "flame",
               scenarios.frame_duration_ms, n_frames=scenarios.n_frames - scenarios.n_unshown)

On the left the flames burn outwards as the edge of a burning sheet of paper does: unburnt mixture pale, burnt mixture dark, smouldering red where it holds more heat than its fuel released, and the burning glowing along the fronts. The three fronts meet and merge, leaving deep cusps where they joined, and the merged front wrinkles into cells, bulges running ahead, parted by creases where the glow fades and which leave dark trails of cooler gas behind them. On the right the energy each parcel holds is red above the mean unburnt mixture's and blue below. A band of deficit runs ahead of the front, where the fuel has spread away faster than the heat has arrived, and the burnt gas behind holds a surplus, largest behind the bulges and least along the creases' trails. Burning moves no energy, so all of that is the work of the two amounts spreading at different rates.

In [ ]:
# checks
# Burning moves energy from the fuel into the heat and makes none.
np.testing.assert_allclose((energy & reaction).to_array(), 0.0, atol=1e-15)
# Over the first frames the disk's energy holds while its heat grows, and the energy a parcel holds falls well below
# what it held at the start ahead of the fronts and rises well above it behind them.
n_checked = 13
early = [later for later, _ in flame(parcels, n_checked)]
energies = np.array([(cells * (energy & later)).sites.sum().to_array() for later in early])
heats = np.array([(cells * (mv.h & later)).sites.sum().to_array() for later in early])
np.testing.assert_allclose(energies, energies[0], rtol=1e-12)
assert np.all(np.diff(heats) > 0)
change = np.asarray((energy & early[-1]).to_array() - (energy & parcels).to_array())
assert change.min() < -0.25 and change.max() > 0.2